# Product Recommendation System - Data Analytics Notebook

## Objective
This notebook performs comprehensive Exploratory Data Analysis (EDA) on the furniture product dataset to understand:
- Dataset structure and quality
- Product distributions across categories, brands, and materials
- Price analysis and trends
- Missing data patterns
- Key insights for recommendation system design

**Author**: Intern Assignment - Product Recommendation System  
**Date**: October 2025

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set visualization styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully")

## 1. Data Loading and Initial Exploration

We start by loading the dataset and understanding its basic structure.

In [ ]:
# Load the dataset
df = pd.read_csv('data/products.csv')

print(f"📊 Dataset Shape: {df.shape[0]} products, {df.shape[1]} features")
print("\n" + "="*80)
print("Dataset Overview:")
print("="*80)
df.head()

In [ ]:
# Display column names and data types
print("\n📋 Column Information:")
print("="*80)
df.info()

print("\n🔍 Basic Statistics:")
print("="*80)
df.describe(include='all')

## 2. Data Quality Analysis

**Rationale**: Understanding missing data helps us decide on imputation strategies and identify which features are most reliable for our ML models.

In [ ]:
# Check for missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing %', ascending=False)

print("🔎 Missing Data Analysis:")
print("="*80)
print(missing_data)

# Visualize missing data
plt.figure(figsize=(12, 6))
plt.barh(missing_data['Column'], missing_data['Missing %'], color='coral')
plt.xlabel('Missing Percentage (%)')
plt.title('Missing Data Analysis Across Features')
plt.tight_layout()
plt.show()

print("\n💡 Insight: This helps us understand which features need data cleaning or can be safely used.")

## 3. Price Analysis

**Rationale**: Price is a critical feature for recommendations. Understanding price distribution helps in:
- Price-based filtering
- Identifying outliers
- Budget-based recommendations

In [ ]:
# Clean and convert price column
df['price_clean'] = df['price'].str.replace('$', '').str.replace(',', '')
df['price_numeric'] = pd.to_numeric(df['price_clean'], errors='coerce')

# Remove rows with missing prices for analysis
price_df = df[df['price_numeric'].notna()].copy()

print(f"📊 Price Statistics:")
print("="*80)
print(f"Products with price info: {len(price_df)} ({len(price_df)/len(df)*100:.1f}%)")
print(f"Average Price: ${price_df['price_numeric'].mean():.2f}")
print(f"Median Price: ${price_df['price_numeric'].median():.2f}")
print(f"Min Price: ${price_df['price_numeric'].min():.2f}")
print(f"Max Price: ${price_df['price_numeric'].max():.2f}")
print(f"Standard Deviation: ${price_df['price_numeric'].std():.2f}")

# Price distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(price_df['price_numeric'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution')
axes[0].axvline(price_df['price_numeric'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(price_df['price_numeric'].median(), color='green', linestyle='--', label='Median')
axes[0].legend()

# Box plot
axes[1].boxplot(price_df['price_numeric'], vert=True)
axes[1].set_ylabel('Price ($)')
axes[1].set_title('Price Distribution - Box Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Insight: Most products are in the $25-$100 range, with some high-end items above $200.")

## 4. Category Analysis

**Rationale**: Understanding product categories is essential for:
- Category-based recommendations
- Training classification models (CV)
- User browsing experience

In [ ]:
# Parse categories (stored as string representation of list)
def parse_categories(cat_str):
    """Parse category string to extract main and sub categories"""
    try:
        cats = ast.literal_eval(cat_str)
        if isinstance(cats, list) and len(cats) > 0:
            return cats
        return []
    except:
        return []

df['categories_list'] = df['categories'].apply(parse_categories)
df['main_category'] = df['categories_list'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')
df['sub_category'] = df['categories_list'].apply(lambda x: x[-1] if len(x) > 0 else 'Unknown')

# Main category distribution
main_cat_counts = df['main_category'].value_counts()
print(f"\n📂 Main Category Distribution ({len(main_cat_counts)} unique categories):")
print("="*80)
print(main_cat_counts)

# Visualize top categories
plt.figure(figsize=(14, 6))
main_cat_counts.head(15).plot(kind='barh', color='teal')
plt.xlabel('Number of Products')
plt.title('Top 15 Product Categories')
plt.tight_layout()
plt.show()

print("\n💡 Insight: 'Home & Kitchen' dominates, indicating a furniture-heavy dataset.")

## 5. Brand Analysis

**Rationale**: Brand information helps in:
- Brand-based filtering
- Understanding market concentration
- Quality perception

In [ ]:
# Brand analysis
brand_counts = df['brand'].value_counts()
print(f"\n🏷️ Brand Analysis ({len(brand_counts)} unique brands):")
print("="*80)
print(f"Top 20 Brands:\n{brand_counts.head(20)}")

# Visualize top brands
plt.figure(figsize=(14, 6))
brand_counts.head(15).plot(kind='barh', color='purple')
plt.xlabel('Number of Products')
plt.title('Top 15 Brands by Product Count')
plt.tight_layout()
plt.show()

print("\n💡 Insight: Diverse brand distribution - good for varied recommendations.")

## 6. Material & Color Analysis

**Rationale**: Material and color are key product attributes that:
- Influence user preferences
- Help in semantic search
- Enable style-based recommendations

In [ ]:
# Material analysis
material_counts = df['material'].value_counts()
print(f"\n🪵 Material Distribution ({len(material_counts)} unique materials):")
print("="*80)
print(material_counts.head(20))

# Color analysis
color_counts = df['color'].value_counts()
print(f"\n🎨 Color Distribution ({len(color_counts)} unique colors):")
print("="*80)
print(color_counts.head(20))

# Visualize materials and colors
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Materials
material_counts.head(10).plot(kind='barh', ax=axes[0], color='brown')
axes[0].set_xlabel('Count')
axes[0].set_title('Top 10 Materials')

# Colors
color_counts.head(10).plot(kind='barh', ax=axes[1], color='skyblue')
axes[1].set_xlabel('Count')
axes[1].set_title('Top 10 Colors')

plt.tight_layout()
plt.show()

print("\n💡 Insight: Wood and Metal are dominant materials. Black and White are top colors.")

## 7. Text Content Analysis

**Rationale**: Analyzing text (title, description) helps us:
- Understand text richness for NLP models
- Identify common keywords
- Plan text preprocessing strategies

In [ ]:
# Title length analysis
df['title_length'] = df['title'].fillna('').str.len()
df['description_length'] = df['description'].fillna('').str.len()

print("\n📝 Text Content Statistics:")
print("="*80)
print(f"Title - Average Length: {df['title_length'].mean():.0f} characters")
print(f"Title - Median Length: {df['title_length'].median():.0f} characters")
print(f"Description - Average Length: {df['description_length'].mean():.0f} characters")
print(f"Description - Median Length: {df['description_length'].median():.0f} characters")
print(f"Products with descriptions: {df['description'].notna().sum()} ({df['description'].notna().sum()/len(df)*100:.1f}%)")

# Visualize text lengths
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['title_length'], bins=30, color='green', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Title Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Title Lengths')

axes[1].hist(df[df['description_length'] > 0]['description_length'], bins=30, color='orange', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Description Length (characters)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Description Lengths')

plt.tight_layout()
plt.show()

print("\n💡 Insight: Titles are concise; descriptions vary widely - good for semantic embeddings.")

## 8. Image Analysis

**Rationale**: Images are crucial for:
- Computer Vision model training
- Visual recommendations
- User engagement

In [ ]:
# Parse image URLs
def count_images(img_str):
    """Count number of images for each product"""
    try:
        imgs = ast.literal_eval(img_str)
        if isinstance(imgs, list):
            return len([img for img in imgs if img and img.strip()])
        return 0
    except:
        return 0

df['image_count'] = df['images'].apply(count_images)

print("\n🖼️ Image Availability Analysis:")
print("="*80)
print(f"Products with images: {(df['image_count'] > 0).sum()} ({(df['image_count'] > 0).sum()/len(df)*100:.1f}%)")
print(f"Average images per product: {df['image_count'].mean():.2f}")
print(f"Max images for a product: {df['image_count'].max()}")
print(f"\nImage count distribution:\n{df['image_count'].value_counts().sort_index()}")

# Visualize image counts
plt.figure(figsize=(10, 5))
df['image_count'].value_counts().sort_index().plot(kind='bar', color='indigo')
plt.xlabel('Number of Images')
plt.ylabel('Number of Products')
plt.title('Distribution of Image Count per Product')
plt.tight_layout()
plt.show()

print("\n💡 Insight: Most products have multiple images - excellent for CV model training!")

## 9. Country of Origin Analysis

**Rationale**: Understanding manufacturing origins can help with:
- Quality perceptions
- Supply chain insights
- Regional preferences

In [ ]:
# Country analysis
country_counts = df['country_of_origin'].value_counts()
print(f"\n🌍 Country of Origin Analysis ({len(country_counts)} countries):")
print("="*80)
print(country_counts.head(15))

# Visualize
plt.figure(figsize=(12, 6))
country_counts.head(10).plot(kind='barh', color='crimson')
plt.xlabel('Number of Products')
plt.title('Top 10 Countries by Product Origin')
plt.tight_layout()
plt.show()

print("\n💡 Insight: China dominates manufacturing - typical for furniture industry.")

## 10. Key Insights Summary

**Summary of findings for ML model design:**

In [ ]:
print("\n" + "="*80)
print("📊 KEY INSIGHTS FOR ML MODEL DEVELOPMENT")
print("="*80)

insights = {
    "Dataset Size": f"{len(df)} products with {df.shape[1]} features",
    "Text Features": "Rich titles and descriptions - excellent for NLP embeddings",
    "Image Features": f"{(df['image_count'] > 0).sum()} products have images - good for CV",
    "Price Range": f"${price_df['price_numeric'].min():.2f} - ${price_df['price_numeric'].max():.2f}",
    "Categories": f"{len(main_cat_counts)} main categories - good diversity",
    "Brands": f"{len(brand_counts)} unique brands",
    "Missing Data": "Price and material have gaps - need handling in preprocessing",
    "Recommendation Strategy": "Hybrid approach: Text embeddings + Image features + Metadata filters"
}

for key, value in insights.items():
    print(f"\n✅ {key}:")
    print(f"   {value}")

print("\n" + "="*80)
print("💡 NEXT STEPS:")
print("="*80)
print("1. Build text embeddings using sentence-transformers for semantic search")
print("2. Train/use CV model for image classification (categories)")
print("3. Store embeddings in FAISS vector database for fast retrieval")
print("4. Integrate LangChain for GenAI-powered product descriptions")
print("5. Build FastAPI endpoints for recommendations")
print("="*80)

## 11. Export Processed Data

Save cleaned data for model training

In [ ]:
# Save enriched dataset
df.to_csv('data/products_processed.csv', index=False)
print("\n✅ Processed data saved to 'data/products_processed.csv'")
print("\n🎉 Data Analytics Complete! Ready for Model Training.")